# 03 — Tiền xử lý hai dataset cho FL 2 client × 3 lớp

Notebook giải thích và khảo sát pipeline **multidataset_v1** (CIC-DDoS2019 + UNSW-NB15).
Mọi xử lý nằm trong Python (`fl_ddos/dataset_adapters.py`, `fl_ddos/multidataset_prep.py`, `fl_ddos/multidataset_eval.py`);
notebook chỉ gọi lại cùng hàm và đọc artifacts do CLI tạo.

**Run All không tải dữ liệu, không chạy pipeline dài, không train.** Các bước đó là lệnh riêng (ô cuối mục 1).
Đổi thư mục bằng biến môi trường `MDS_DATA_DIR`, `MDS_RESULTS_DIR` (mặc định profile smoke).

In [ ]:
import json, os, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
from fl_ddos import dataset_adapters as ad, multidataset_prep as mp

CFG = mp.load_config(ROOT / "configs/multidataset.json")
DATA = Path(os.environ.get("MDS_DATA_DIR", ROOT / "data_paper_multi/smoke"))
RESULTS = Path(os.environ.get("MDS_RESULTS_DIR", ROOT / "results_multidataset/smoke"))
HAVE_DATA = (DATA / "features.json").exists()
HAVE_RESULTS = (RESULTS / "evaluation_by_dataset.csv").exists()
print("artifacts:", DATA, "->", "có" if HAVE_DATA else "CHƯA CÓ")
print("kết quả  :", RESULTS, "->", "có" if HAVE_RESULTS else "CHƯA CÓ")

def need(flag, what):
    if not flag:
        print(f"(bỏ qua: chưa có {what}; xem lệnh ở mục 1)")
    return flag

## 1. Hai nguồn, hai client, ba lớp

| Client | partition-id | Dataset | Ghi chú |
|---|---:|---|---|
| 1 | 0 | CIC-DDoS2019 | 17 Parquet; `*-testing` làm train pool, `*-training` làm test (quy ước hiện tại) |
| 2 | 1 | UNSW-NB15 | CSV training-set / testing-set chính thức |

Đầu ra mô hình: **0 = BENIGN, 1 = DOS_DDOS, 2 = OTHER_ATTACK** — chỉ đến cấp nhóm. DOS_DDOS gộp DoS (UNSW) và DDoS
(CIC), nên không được đọc thành "DDoS thuần". Đây là điều chỉnh của đồ án so với 5 lớp ở Hình 14 của paper.
OTHER_ATTACK chỉ có ở UNSW-NB15; client CIC không có lớp này (class weight = 0).

IoT-23 đã bị loại khỏi hệ thống ngày 09/11/2026: chia theo capture làm phân phối train/val/test lệch nặng
(tỷ lệ thiếu `duration` 59% / 98% / 93%), mô hình gần như không phát hiện được tấn công trên test IoT-23.

Lệnh chạy (thực hiện riêng, không chạy trong notebook):

```powershell
.\.venv\Scripts\python.exe prepare/download_datasets.py                       # CIC kiểm tra, UNSW ~vài chục MB
.\.venv\Scripts\python.exe prepare/multidataset_pipeline.py --profile smoke
.\scripts\fl_run.ps1 "scenario='multidataset' num-server-rounds=2 local-epochs=1 seed=0" -Clients 2 -Actors 1 -DataDir data_paper_multi/smoke -ResultsDir results_multidataset/smoke
.\.venv\Scripts\python.exe analysis/evaluate_multidataset.py --data-dir data_paper_multi/smoke --results-dir results_multidataset/smoke --checkpoint "results_multidataset/smoke/fl/*_best.pt"
```

In [ ]:
pd.DataFrame(CFG["clients"])

## 2. Hiện trạng file nguồn

In [ ]:
rows = []
for ds, src in CFG["sources"].items():
    d = ROOT / src["dir"]
    files = sorted(p.name for p in d.glob("*") if p.is_file()) if d.exists() else []
    mf = d / "download_manifest.json"
    rows.append({"dataset": ds, "thư mục": str(d.relative_to(ROOT)), "số file": len(files),
                 "manifest": "có" if mf.exists() else "chưa", "ví dụ": ", ".join(files[:3])})
pd.DataFrame(rows)

In [ ]:
cic = ROOT / CFG["sources"]["cicddos2019"]["dir"]
if need(any(cic.glob("*.parquet")), "CIC Parquet"):
    f = sorted(cic.glob("*.parquet"))[0]
    raw = pd.read_parquet(f)
    print(f.name, raw.shape, "cột:", len(raw.columns))
    print("Nhãn gốc theo file:")
    display(pd.DataFrame({p.stem: pd.read_parquet(p, columns=["Label"])["Label"].value_counts()
                          for p in sorted(cic.glob("*.parquet"))}).fillna(0).astype(int).T)

## 3. Ánh xạ nhãn

Bảng cố định trong `configs/multidataset.json`. Không dùng substring tùy ý; không đổi Portmap → PortScan.
Nhãn lạ/mâu thuẫn không bao giờ được gán mặc định; pipeline dừng.

In [ ]:
lm = CFG["label_map"]
for ds in ("cicddos2019", "unsw_nb15"):
    print(ds); display(pd.DataFrame([(c, ", ".join(v)) for c, v in lm[ds].items()], columns=["lớp", "nhãn gốc"]))

In [ ]:
if need(HAVE_DATA, "artifacts"):
    q = json.load(open(DATA / "quality_report.json", encoding="utf-8"))
    for key, title in (("full_counts", "Số dòng hợp lệ gốc theo split"), ("used_counts", "Số dòng dùng (subset)")):
        t = pd.concat({ds: pd.DataFrame(v).T for ds, v in q[key].items()}).fillna(0).astype(int)
        print(title); display(t)
    print("Policy:", q.get("sampling_policy", "legacy"))
    if q.get("integration_only"):
        print("SMOKE: chỉ kiểm tra tích hợp, không dùng để kết luận chất lượng mô hình.")
    for warning in q.get("warnings", []):
        print("Cảnh báo:", warning)
    plans = [dict(dataset=ds, split=sp, **{k: v for k, v in plan.items() if k not in ("quota", "floor_shortfall")})
             for ds, splits in q.get("sampling", {}).items() for sp, plan in splits.items()]
    if plans:
        print("Ngân sách và support thực dùng:"); display(pd.DataFrame(plans))


## 4. Ánh xạ đặc trưng

Schema v1 chỉ gồm 4 trường cơ sở có ý nghĩa tương đương đã kiểm chứng; byte, IAT, window, flag, cổng không dùng.

In [ ]:
pd.DataFrame({"trường chuẩn": ["duration_s", "fwd_packets", "bwd_packets", "protocol_group"],
              "CIC": ["Flow Duration / 1e6 (µs -> s)", "Total Fwd Packets", "Total Backward Packets", "Protocol (6/17/1)"],
              "UNSW": ["dur (s)", "spkts", "dpkts", "proto"]})

In [ ]:
print("8 cột liên tục ứng viên DAS:", mp.CONTINUOUS)
print("6 cột cố định:", mp.FIXED)
demo = pd.DataFrame({"duration_s": [1.0, 0.0, np.nan], "fwd_packets": [3, 1, 2], "bwd_packets": [1, 0, 2],
                     "protocol_group": ["tcp", "udp", "icmp"]})
mp.derive(demo)  # duration 0/missing -> rate missing, không Inf

## 5. Chất lượng, missing, split và overlap

In [ ]:
if need(HAVE_DATA, "artifacts"):
    print("Kiểm tra tương thích:"); display(pd.DataFrame([dict(c, dataset=ds) for ds, cs in q["compatibility_checks"].items() for c in cs]))
    print("Ràng buộc độc lập:"); display(pd.DataFrame(q["independence"]).T)
    print("Tỷ lệ duration missing:"); display(pd.DataFrame(q["missing_duration"]).T)
    drops = pd.DataFrame([dict(r, dataset=ds) for ds, rs in q["drops"].items() for r in rs])
    print("Dòng bị loại:"); display(drops if len(drops) else "không có")
    man = json.load(open(DATA / "dataset_manifest.json", encoding="utf-8"))
    display(pd.DataFrame({ds: s["split"] for ds, s in man["sources"].items()}).T)

## 6. DAS: CRA + IDF, một danh sách cho cả federation

Relevance pooled tính từ thống kê train của hai client (θ = 0,001, γ = 0,9, tối đa 8 cột liên tục). Cột relevance
theo nguồn chỉ để nhìn ảnh hưởng phân phối; danh sách cuối dùng chung cho mọi client: liên tục theo thứ tự DAS + 6 cột cố định.

In [ ]:
if need(HAVE_DATA, "artifacts"):
    fs = pd.read_csv(DATA / "feature_selection.csv"); display(fs)
    feats = json.load(open(DATA / "features.json", encoding="utf-8"))
    print(f"F = {len(feats['features'])}:", feats["features"])
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
    for ax, k in zip(axes, (1, 2)):
        df = pd.read_parquet(DATA / "multidataset" / f"client_{k}.parquet")
        for c in feats["continuous_selected"][:4]:
            ax.hist(df[c], bins=40, alpha=.5, label=c, histtype="stepfilled")
        ax.set_title(f"client {k} ({df['dataset_id'].iloc[0]}) sau AFN", fontsize=9)
    axes[0].legend(fontsize=7); plt.tight_layout(); plt.show()

## 7. Kết quả FL / test theo nguồn

Một best checkpoint (chọn bằng TB đều F1-macro-present trên hai validation) được đánh giá một lần trên hai test,
ngưỡng 0,5 cố định. `attack_*`: phát hiện mọi tấn công (1 − P(BENIGN)); `dos_*`: phát hiện nhóm DoS/DDoS (P(DOS_DDOS)).
N/A = metric không xác định do thiếu một nhãn (ví dụ OTHER_ATTACK ở CIC), không thay bằng 0 hoặc 1.
`pooled_predictions` tính từ dự đoán gộp, khác `mean_over_sources`. Subset test không phải toàn bộ benchmark.

In [ ]:
if need(HAVE_RESULTS, "kết quả đánh giá"):
    ev = pd.read_csv(RESULTS / "evaluation_by_dataset.csv")
    display(ev[["method", "seed", "selected_round", "dataset_id", "n", "acc", "f1_macro_present",
                "attack_f1", "attack_far_benign", "attack_roc_auc", "dos_f1", "dos_roc_auc"]])
    display(pd.read_csv(RESULTS / "per_class.csv")[["method", "seed", "dataset_id", "class", "support", "precision", "recall", "f1"]])
    for f in sorted((RESULTS / "figures").glob("confusion_*.png"))[:3]:
        from IPython.display import Image; display(Image(filename=str(f)))

In [ ]:
if need(HAVE_RESULTS, "kết quả đánh giá"):
    po = pd.read_csv(RESULTS / "per_original_label.csv")
    print("Phân tích theo nhãn gốc (không phải dự đoán loại con):")
    display(po[["method", "dataset_id", "original_label", "true_class", "n", "attack_detect_rate", "dos_detect_rate"]])

**Giới hạn cần ghi trong báo cáo:** schema chung chỉ 4 trường cơ sở; DOS_DDOS gộp DoS và DDoS; OTHER_ATTACK là
nhóm theo hệ nhãn nguồn; quy tắc tạo/đóng flow của hai extractor (CICFlowMeter, Argus) khác nhau; subset theo quota; không tái lập
nguyên vẹn thiết lập hoặc điểm số của paper.